# LFC density sweep (ZEMAX HDF / `prf_convolution`)

Minimal notebook: paint unresolved LFC lines onto BSPEC fiber 2 with the default
ZEMAX HDF model (`HISPEC_yJ_20231006A-1_all_traces.hdf`) via
`SimulateRateMap` → `prf_convolution`.

Lines are placed evenly in **spectral-row** space with a golden-ratio phase offset
per echelle order so features do not line up vertically across orders. Densities:
`50, 100, 200, 400, 800` lines/order. Each 2D rate map is saved under `OUT`.


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
from astropy.io import fits

import hispecdrp
from hispecdrp.simulations.e2e_pipeline import (
    get_sim_hdf_trace_grids,
    paint_injected_emission_comb,
)
from hispecdrp.simulations.simulate_rate_map import SimulateRateMap

OUT = Path("/Users/ckilpatrick/hispec_lfc_density_sweep")
OUT.mkdir(parents=True, exist_ok=True)

SPECTROGRAPH = "BSPEC"
FIBER = 2
N_LINES_LIST = [50, 100, 200, 400, 800]
EDGE_MARGIN = 0.05
PHI = (np.sqrt(5.0) - 1.0) / 2.0  # golden-ratio conjugate for irrational stagger
PAINT_METHOD = "prf_convolution"
PHOTONS_PER_LINE = 1.0e12

srm = SimulateRateMap()
hdf_path = srm.hdf_file_directory + srm.bspec_hdf_model
print(f"hispecdrp {hispecdrp.__version__}")
print(f"HDF: {hdf_path}")
print(f"OUT: {OUT}")
assert Path(hdf_path).is_file(), f"missing ZEMAX HDF: {hdf_path}"


In [ ]:
grids = get_sim_hdf_trace_grids(SPECTROGRAPH)
orders = sorted(grids[FIBER].keys())
pix = np.arange(4096, dtype=np.float64)


def build_staggered_lfc_line_list(n_lines: int, edge: float = EDGE_MARGIN) -> dict[int, np.ndarray]:
    """Even spacing in spectral-row space; golden-ratio phase offset per order."""
    out: dict[int, np.ndarray] = {}
    n = int(n_lines)
    for i, od in enumerate(orders):
        w = np.asarray(grids[FIBER][od]["wavelength"], dtype=np.float64)
        ok = np.isfinite(w)
        lo, hi = np.percentile(pix[ok], [100.0 * edge, 100.0 * (1.0 - edge)])
        phase = (i * PHI) % 1.0
        rows = lo + (np.arange(n, dtype=np.float64) + phase) * (hi - lo) / n
        out[int(od)] = np.interp(rows, pix, w)
    return out


def stagger_report(line_list: dict[int, np.ndarray], n_lines: int) -> dict:
    """Nearest-neighbor |Δrow| between adjacent orders (from intended row samples)."""
    rows_by = {}
    for i, od in enumerate(orders):
        w = np.asarray(grids[FIBER][od]["wavelength"], dtype=np.float64)
        ok = np.isfinite(w)
        lo, hi = np.percentile(pix[ok], [100.0 * EDGE_MARGIN, 100.0 * (1.0 - EDGE_MARGIN)])
        phase = (i * PHI) % 1.0
        rows_by[od] = lo + (np.arange(n_lines, dtype=np.float64) + phase) * (hi - lo) / n_lines
    d = []
    for a, b in zip(orders[:-1], orders[1:]):
        ra, rb = rows_by[a], rows_by[b]
        for r in ra:
            d.append(float(np.min(np.abs(rb - r))))
    d = np.asarray(d, dtype=np.float64)
    spacing = float(np.median([rows_by[od][1] - rows_by[od][0] for od in orders]))
    return {
        "n_lines": int(n_lines),
        "median_row_spacing_pix": spacing,
        "nn_drow_min_pix": float(d.min()),
        "nn_drow_med_pix": float(np.median(d)),
        "nn_drow_p10_pix": float(np.percentile(d, 10)),
        "frac_nn_lt_0p5_pix": float(np.mean(d < 0.5)),
    }


# Verify stagger before any paint/save
probe = build_staggered_lfc_line_list(50)
rep = stagger_report(probe, 50)
print("stagger check (N=50):", json.dumps(rep, indent=2))
assert rep["frac_nn_lt_0p5_pix"] == 0.0, "lines still align across orders within 0.5 pix"
assert rep["nn_drow_min_pix"] > 0.2 * rep["median_row_spacing_pix"], (
    "stagger too weak vs line spacing"
)
print("OK: lines are staggered in spectral-row space across orders.")


In [ ]:
manifest = {
    "spectrograph": SPECTROGRAPH,
    "fiber": FIBER,
    "hdf": hdf_path,
    "paint_method": PAINT_METHOD,
    "photons_per_line": PHOTONS_PER_LINE,
    "edge_margin": EDGE_MARGIN,
    "phase": "golden_ratio_row_space",
    "n_lines_list": N_LINES_LIST,
    "frames": [],
}

for n in N_LINES_LIST:
    print(f"\n=== N={n} lines/order ===")
    line_list = build_staggered_lfc_line_list(n)
    rep = stagger_report(line_list, n)
    print(
        f"  stagger: med spacing={rep['median_row_spacing_pix']:.2f} pix  "
        f"nn|Δrow| min/med={rep['nn_drow_min_pix']:.3f}/{rep['nn_drow_med_pix']:.3f}  "
        f"frac<0.5pix={rep['frac_nn_lt_0p5_pix']:.3f}"
    )
    assert rep["frac_nn_lt_0p5_pix"] == 0.0

    rate = paint_injected_emission_comb(
        line_list,
        spectrograph=SPECTROGRAPH,
        fiber=FIBER,
        method=PAINT_METHOD,
        photons_per_line=PHOTONS_PER_LINE,
    )
    rate = np.asarray(rate, dtype=np.float64)

    tag = f"n{n:04d}"
    rate_path = OUT / f"lfc_rate_BSPEC_fib{FIBER}_{tag}.fits"
    lines_path = OUT / f"lfc_lines_BSPEC_fib{FIBER}_{tag}.npz"
    fits.PrimaryHDU(
        rate.astype(np.float32),
        header=fits.Header(
            {
                "SPEC": SPECTROGRAPH,
                "FIBER": FIBER,
                "NLINES": n,
                "METHOD": PAINT_METHOD,
                "HDF": Path(hdf_path).name,
            }
        ),
    ).writeto(rate_path, overwrite=True)

    # sidecar wavelengths (µm) per order
    np.savez(
        lines_path,
        **{f"order_{od}": np.asarray(lams, dtype=np.float64) for od, lams in line_list.items()},
    )

    entry = {
        "n_lines_per_order": n,
        "n_orders": len(line_list),
        "n_lines_total": int(sum(np.asarray(v).size for v in line_list.values())),
        "rate_path": str(rate_path),
        "lines_path": str(lines_path),
        "rate_sum": float(rate.sum()),
        "rate_max": float(rate.max()),
        "stagger": rep,
    }
    manifest["frames"].append(entry)
    print(f"  saved {rate_path.name}  shape={rate.shape}  max={rate.max():.3e}  sum={rate.sum():.3e}")

manifest_path = OUT / "lfc_density_sweep_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(f"\nwrote {manifest_path}")
print(json.dumps({k: manifest[k] for k in manifest if k != "frames"}, indent=2))
for fr in manifest["frames"]:
    print(f"  N={fr['n_lines_per_order']}: {fr['rate_path']}")
